
## 1. Load data

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # add project root to path

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.data.data_ingestion  import DataIngestion
from src.data.data_preprocessing  import DataPreprocessor
from src.features.build_features   import FeatureEngineer
#from src.features.feature_selection import FeatureSelector
from src.visualization.eda_plots   import EDAVisualizer


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

viz    = EDAVisualizer()

In [3]:
ingestor = DataIngestion(config_path="../config.yaml")
df = ingestor.load(path="../data/train-test.csv")
target = ingestor.target


2026-09-05 23:23:56 | INFO     | src.data.data_ingestion | Loading data from: E:\Python projects\Spotter_Machine Learning Engineer_Assessment\data\train-test.csv
2026-09-05 23:23:56 | INFO     | src.data.data_ingestion | Loaded  shape  : (48000, 14)
2026-09-05 23:23:56 | INFO     | src.data.data_ingestion | Memory usage   : 19.23 MB
2026-09-05 23:23:56 | INFO     | src.data.data_ingestion | Target 'posted_rate' — Mean: 2,373.98



## 2. Split FIRST — Before Any Fitting

The goal is to predict future rates using past data. A random split may cause temporal leakage, so I chose to use a chronological split.



In [7]:
# Sort by date first
df = df.sort_values("date").reset_index(drop=True)

test_split_idx = int(len(df) * 0.80)

train_val_df = df.iloc[:test_split_idx].copy()
test_df = df.iloc[test_split_idx:].copy()

val_split_idx = int(len(train_val_df) * 0.80)

train_df = train_val_df.iloc[:val_split_idx].copy()
val_df = train_val_df.iloc[val_split_idx:].copy()

X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_val = val_df.drop(columns=[target])
y_val = val_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_val  :", y_val.shape)
print("y_test :", y_test.shape)


X_train: (30720, 13)
X_val  : (7680, 13)
X_test : (9600, 13)
y_train: (30720,)
y_val  : (7680,)
y_test : (9600,)


---
## 3. Data Preprocessing — Fit on Train Only


In [12]:
preprocessor = DataPreprocessor(config_path="../config.yaml")

train_clean = preprocessor.fit_transform(X_train)
val_clean = preprocessor.transform(X_val)
test_clean  = preprocessor.transform(X_test)

print("=== Cleaning report (train set) ===")
for k, v in preprocessor.cleaning_report.items():
    print(f"  {k:<25} : {v}")

print(f"\nTrain shape before : {X_train.shape}")
print(f"Train shape after  : {train_clean.shape}")

print(f"\nValidation shape before : {X_val.shape}")
print(f"Validation shape after  : {val_clean.shape}")

print(f"\nTest shape before : {X_test.shape}")
print(f"Test shape after  : {test_clean.shape}")

print(f"\ny_train shape : {y_train.shape}")
print(f"y_val shape   : {y_val.shape}")
print(f"y_test shape  : {y_test.shape}")


2026-09-06 02:44:46 | INFO     | src.data.data_preprocessing | === Data Preprocessing START ===
2026-09-06 02:44:46 | INFO     | src.data.data_preprocessing | date Dtype fixes applied.
2026-09-06 02:44:46 | INFO     | src.data.data_preprocessing | Handling 432 missing values (numerical: median, categorical: mode)
2026-09-06 02:44:46 | INFO     | src.data.data_preprocessing | Preprocessing complete. 30,720 → 30,720 rows (0 removed)
2026-09-06 02:44:46 | INFO     | src.data.data_preprocessing | date Dtype fixes applied.
2026-09-06 02:44:46 | INFO     | src.data.data_preprocessing | Handling 102 missing values (numerical: median, categorical: mode)
2026-09-06 02:44:46 | INFO     | src.data.data_preprocessing | date Dtype fixes applied.
2026-09-06 02:44:46 | INFO     | src.data.data_preprocessing | Handling 140 missing values (numerical: median, categorical: mode)


=== Cleaning report (train set) ===
  original_shape            : (30720, 13)
  clean_shape               : (30720, 13)
  rows_removed              : 0

Train shape before : (30720, 13)
Train shape after  : (30720, 13)

Validation shape before : (7680, 13)
Validation shape after  : (7680, 13)

Test shape before : (9600, 13)
Test shape after  : (9600, 13)

y_train shape : (30720,)
y_val shape   : (7680,)
y_test shape  : (9600,)



## 4. Feature Engineering — Fit on Train Only


In [8]:
# Separate features from target — feature engineer never sees the target
X_train_clean = train_clean.drop(columns=[target], errors="ignore")
X_val_clean   = val_clean.drop(columns=[target], errors="ignore")
X_test_clean  = test_clean.drop(columns=[target], errors="ignore")

fe = FeatureEngineer("../config.yaml")

# FIT on training features only
X_train_eng = fe.fit_transform(X_train_clean)

# APPLY to val and test — uses transformer learned from train
X_val_eng = fe.transform(X_val_clean)
X_test_eng  = fe.transform(X_test_clean)

print(f"Features before engineering : {X_train_clean.shape[1]}")
print(f"Features after  engineering : {X_train_eng.shape[1]}")

print("\nValidation features after engineering :", X_val_eng.shape[1])
print("Test features after engineering       :", X_test_eng.shape[1])

print(f"\nNew engineered columns:")
for col in dict.fromkeys(fe._engineered_cols):
    print(f"  {col}")

2026-09-05 23:09:22 | INFO     | src.features.build_features | === Feature Engineering FIT ===
2026-09-05 23:09:22 | INFO     | src.features.build_features | Time features created: ['year', 'month', 'day_of_week', 'is_weekend']
2026-09-05 23:09:22 | INFO     | src.features.build_features | Distance features created: ['distance_log']
2026-09-05 23:09:22 | INFO     | src.features.build_features | Categorical encoding applied to 4 column(s).
2026-09-05 23:09:22 | INFO     | src.features.build_features | Feature engineering fitted. Output features: 17
2026-09-05 23:09:22 | INFO     | src.features.build_features | === Feature Engineering TRANSFORM ===
2026-09-05 23:09:22 | INFO     | src.features.build_features | Time features created: ['year', 'month', 'day_of_week', 'is_weekend']
2026-09-05 23:09:22 | INFO     | src.features.build_features | Distance features created: ['distance_log']
2026-09-05 23:09:22 | INFO     | src.features.build_features | Categorical encoding applied to 4 column(s

Features before engineering : 13
Features after  engineering : 17

Validation features after engineering : 17
Test features after engineering       : 17

New engineered columns:
  year
  month
  day_of_week
  is_weekend
  distance_log


### 5.1 Spot-check engineered features (train set)

In [9]:
new_cols = [c for c in dict.fromkeys(fe._engineered_cols) if c in X_train_eng.columns]
print("=== New features — first 5 rows ===")
print(X_train_eng[new_cols].head())

=== New features — first 5 rows ===
   year  month  day_of_week  is_weekend  distance_log
0  2025      1            2           0        5.6179
1  2025      1            2           0        6.5406
2  2025      1            2           0        7.0022
3  2025      1            2           0        6.5296
4  2025      1            2           0        6.7941
